# **R/S BENCHMARK - DATASET GENERATION**

## **1. State limit function**

$$g = k(t) \cdot \frac{R}{z_1} - S \cdot z_2$$

$z_1$ and $z_2$ are normal latent multipliers (mean 1.0, sd 0.028 and 0.096); $k(t) = 1 +
(k_{final}-1)\,t/100$ is the degradation factor.

## **2. Libraries**

In [1]:
import sys
import time
from pathlib import Path

# functions.py sits one directory up
sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Normal, JointIndependent

/home/casa-wand/steam2tb/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **3. Random variables and fixed parameters**

Design variables $R$ and $S$, plus everything the emulator needs that isn't a design variable.

In [2]:
r_mean = 5.0   # resistance mean
r_std  = 0.8   # resistance standard deviation
s_mean = 2.0   # load mean
s_std  = 0.6   # load standard deviation

n_samples            = 2000     # Number of design samples
n_latent_samples     = 2500     # Number of latent samples per design sample. Also the filename prefix
n_samples_validation = 500      # Number of validation samples, redrawn at every time step
n_lambdas            = 4        # Number of λs (λ1, λ2, λ3, λ4)
k_factor_final       = 0.5      # Degradation factor at t = t_final. Use 1.0 for no time effect
t_final              = 100.0    # Time at which k reaches k_factor_final; held constant afterwards
z1_std               = 0.028    # Standard deviation of the resistance latent multiplier
z2_std               = 0.096    # Standard deviation of the load latent multiplier

times = np.linspace(0, t_final, 5, endpoint=True)  # Time points for the degradation factor
times

array([  0.,  25.,  50.,  75., 100.])

## **4. Design samples**

In [3]:
r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

x_pce_rvs = joint.rvs(n_samples)
x_val     = joint.rvs(n_samples_validation)

print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations per time step: {(n_samples + n_samples_validation) * n_latent_samples}")

Samples generated successfully!
   Number of design samples: 2000
   Number of latent samples per design sample: 2500
   Total simulations per time step: 6250000


In [4]:
x_pce_rvs

array([[5.39427868, 2.30543071],
       [3.71655097, 2.0537216 ],
       [4.4959986 , 2.2554129 ],
       ...,
       [5.62801855, 1.89202406],
       [4.25012757, 2.06795302],
       [5.29993497, 2.56802247]])

## **5. Generate the dataset at each time step**

Steps:

- $g$ evaluation;
- GLD fit; and
- saving `dataset_full`/`dataset_unique` for both splits.

In [5]:
print("="*60)
print("GENERATING THE BENCHMARK DATASET")
print("="*60)

generation_results = []
for t in times:
    result = generate_dataset_at_time_benchmark(
                                                   x_train=x_pce_rvs,
                                                   x_val=x_val,
                                                   time_step=t,
                                                   n_latent_samples=n_latent_samples,
                                                   k_factor_final=k_factor_final,
                                                   t_final=t_final,
                                                   z1_std=z1_std,
                                                   z2_std=z2_std,
                                                   output_dir='.',
                                               )
    generation_results.append(result)

GENERATING THE BENCHMARK DATASET

----------------------------------------
GENERATING DATASET FOR TIME STEP: 0.0 years
----------------------------------------
  train: 2000 design points, 936.21 s total
  val: 500 design points, 214.21 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 25.0 years
----------------------------------------
  train: 2000 design points, 954.88 s total
  val: 500 design points, 256.61 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 50.0 years
----------------------------------------
  train: 2000 design points, 905.88 s total
  val: 500 design points, 233.14 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 75.0 years
----------------------------------------
  train: 2000 design points, 870.40 s total
  val: 500 design points, 219.70 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 100.0 years
----------------------------

In [6]:
with open(f'{n_latent_samples}_dataset_full_train_0.0_benchmark.pkl', 'rb') as f:
    obj = dill.load(f)
obj.head()

,r,s,z1_latent,R_effective,z2_latent,S_effective,k factor,Time (years),g,lambda 1,lambda 2,lambda 3,lambda 4,Processing time (s)
0,5.394279,2.305431,1.003399,5.376006,0.855297,1.971827,1.0,0.0,3.404179,3.097621,5.691298,0.121094,0.094741,0.21884
1,5.394279,2.305431,0.995530,5.418502,1.026650,2.366870,1.0,0.0,3.051632,3.097621,5.691298,0.121094,0.094741,0.21884
2,5.394279,2.305431,1.028229,5.246183,1.127519,2.599416,1.0,0.0,2.646767,3.097621,5.691298,0.121094,0.094741,0.21884
3,5.394279,2.305431,1.041455,5.179562,1.044494,2.408009,1.0,0.0,2.771554,3.097621,5.691298,0.121094,0.094741,0.21884
4,5.394279,2.305431,0.971617,5.551856,1.013633,2.336861,1.0,0.0,3.214995,3.097621,5.691298,0.121094,0.094741,0.21884


## **6. Timing summary**

Cost of building the dataset, per time step.

In [7]:
timing_rows = []
for result in generation_results:
    train_t = result['df_unique_train']['Processing time (s)']
    val_t   = result['df_unique_val']['Processing time (s)']
    timing_rows.append({
                           'Time (years)':   result['time_step'],
                           'n_train':        len(train_t),
                           'Train total (s)': train_t.sum(),
                           'Train mean (ms)': train_t.mean() * 1e3,
                           'n_val':          len(val_t),
                           'Val total (s)':  val_t.sum(),
                       })

emulator_timing = pd.DataFrame(timing_rows)

with open(f'{n_latent_samples}_emulator_timing_benchmark.pkl', 'wb') as f:
    dill.dump(emulator_timing, f)

train_total = emulator_timing['Train total (s)'].sum()
val_total   = emulator_timing['Val total (s)'].sum()

print(f"Train split - eg-value dataset generation time: {train_total:.1f} s")
print(f"Val split   - g-value dataset generation time: {val_total:.1f} s")
print(f"Total g-value dataset generation time (train + val): {train_total + val_total:.1f} s")
emulator_timing

Train split - eg-value dataset generation time: 4466.4 s
Val split   - g-value dataset generation time: 1128.7 s
Total g-value dataset generation time (train + val): 5595.1 s


,Time (years),n_train,Train total (s),Train mean (ms),n_val,Val total (s)
0,0.0,2000,936.209381,468.104691,500,214.212248
1,25.0,2000,954.884701,477.442351,500,256.613263
2,50.0,2000,905.878127,452.939064,500,233.138237
3,75.0,2000,870.398033,435.199017,500,219.697860
4,100.0,2000,798.989566,399.494783,500,205.036812


## **7. Unique dataset statistics**

Concatenate the `dataset_unique` (train + val, all time steps) frames already held in `generation_results` and describe the columns, including the fitted lambdas.

In [8]:
dataset_unique_all = pd.concat(
    [
        result[f'df_unique_{split}'].assign(split=split)
        for result in generation_results
        for split in ('train', 'val')
    ],
    ignore_index=True,
)

print(f"Combined unique dataset: {len(dataset_unique_all)} rows "
      f"({len(generation_results)} time steps x train/val splits)")
dataset_unique_all.describe()

Combined unique dataset: 12500 rows (5 time steps x train/val splits)


,r,s,lambda 1,lambda 2,lambda 3,lambda 4,Processing time (s)
count,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000
mean,5.043878,1.997460,1.787230,6.967538,0.142157,0.131525,0.447605
std,0.800408,0.602259,1.241995,1.916814,0.021126,0.021188,0.581240
min,2.075805,-0.095408,-1.942688,3.556297,0.064353,0.031246,0.160341
25%,4.502607,1.584629,0.867043,5.669739,0.127731,0.117199,0.168138
50%,5.038166,2.000763,1.724148,6.591905,0.141704,0.131525,0.171500
75%,5.603505,2.402173,2.636142,7.806816,0.156122,0.145766,0.213231
max,7.780369,3.908499,6.869090,27.968291,0.226997,0.203234,3.894066
